In [ ]:
import pandas as pd
import requests
import time
from datetime import datetime
from typing import List, Optional, Dict
import warnings
warnings.filterwarnings('ignore')

# 1. ВЫБОР АКЦИЙ (тикеры Мосбиржи)
TICKERS = [
    'SBER',    # Сбербанк
    'T',       # Т-технологии
    'LKOH',    # Лукойл
    'YNDX',    # Яндекс
    'PLZL',    # Полюс
    'ROSN',    # Роснефть
    'NVTK',    # Новатэк
    'NLMK',    # НЛМК
    'SVCB',    # Совкомбанк
    'HEAD',    # Headhunter
    'MOEX'
]

# 2. ВРЕМЕННОЙ ДИАПАЗОН
START_DATE = '2024-01-01'  # Дата начала
END_DATE = '2024-12-31'    # Дата окончания

# 3. ПАРАМЕТРЫ API
BOARD = 'TQBR'              # Режим торгов ('TQBR' - основной режим)
TIMEOUT = 30                # Таймаут запроса в секундах
RETRY_ATTEMPTS = 3          # Количество повторных попыток при ошибке
DELAY_BETWEEN_REQUESTS = 0.5  # Задержка между запросами в секундах

# 4. ИНДЕКС
INDEX_TICKER = 'IMOEX'

# -------------------------------------

class MOEXParser:
    
    BASE_URL = "https://iss.moex.com/iss"
    
    def __init__(self, timeout: int = 30, retry_attempts: int = 3, delay: float = 0.5):

        self.timeout = timeout
        self.retry_attempts = retry_attempts
        self.delay = delay
    
    def get_stock_history(self, 
                         ticker: str, 
                         start_date: str, 
                         end_date: str,
                         board: str = 'TQBR') -> pd.DataFrame:

        url = f"{self.BASE_URL}/history/engines/stock/markets/shares/securities/{ticker}.json"
        
        all_data = []
        start_index = 0
        
        print(f"   Загрузка {ticker}...", end=' ', flush=True)
        
        while True:
            params = {
                'from': start_date,
                'till': end_date,
                'start': start_index,
            }
            
            # Повторные попытки при ошибке
            success = False
            for attempt in range(self.retry_attempts):
                try:
                    response = requests.get(url, params=params, timeout=self.timeout)
                    response.raise_for_status()
                    data = response.json()
                    success = True
                    break
                except Exception as e:
                    if attempt == self.retry_attempts - 1:
                        print(f"✗ Ошибка после {self.retry_attempts} попыток: {e}")
                        return pd.DataFrame()
                    time.sleep(2)
            
            if not success:
                return pd.DataFrame()
            
            # Проверка наличия данных
            if 'history' not in data or not data['history']['data']:
                break
            
            columns = data['history']['columns']
            rows = data['history']['data']
            
            all_data.extend(rows)
            
            if len(rows) < 100:
                break
            
            start_index += 100
            time.sleep(self.delay)  # Задержка между запросами
        
        if not all_data:
            print(f"✗ Нет данных")
            return pd.DataFrame()
        
        # Создание DataFrame
        df = pd.DataFrame(all_data, columns=columns)
        
        # Фильтрация по режиму торгов
        df = df[df['BOARDID'] == board]
        
        if df.empty:
            print(f"✗ Нет данных для режима {board}")
            return pd.DataFrame()
        
        # Обработка данных
        df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
        df['CLOSE'] = pd.to_numeric(df['CLOSE'], errors='coerce')
        
        # Оставляем только нужные колонки
        df = df[['TRADEDATE', 'CLOSE']].dropna()
        df = df.sort_values('TRADEDATE')
        df = df.drop_duplicates(subset=['TRADEDATE'], keep='last')
        
        # Переименовываем колонки
        df = df.rename(columns={'TRADEDATE': 'Date', 'CLOSE': 'Close'})
        df = df.set_index('Date')
        
        print(f"✓ {len(df)} дней")
        
        return df
    
    def get_stocks_history(self, 
                          tickers: List[str], 
                          start_date: str, 
                          end_date: str,
                          board: str = 'TQBR') -> pd.DataFrame:
        all_prices = {}
        
        print(f"\n📥 Загрузка данных с Мосбиржи:")
        print(f"   Период: {start_date} - {end_date}")
        print(f"   Тикеры: {', '.join(tickers)}")
        print("-" * 80)
        
        for ticker in tickers:
            df = self.get_stock_history(ticker, start_date, end_date, board)
            
            if not df.empty:
                all_prices[ticker] = df['Close']
        
        if not all_prices:
            print("\nНе удалось загрузить ни одного тикера")
            return pd.DataFrame()
        
        # Объединение всех данных
        result_df = pd.DataFrame(all_prices)
        
        # Удаление строк с пропусками (дни когда не все акции торговались)
        result_df = result_df.dropna()
        
        print("-" * 80)
        print(f"Загружено {len(result_df)} торговых дней для {len(result_df.columns)} акций")
        print(f"Период фактически: {result_df.index.min().strftime('%Y-%m-%d')} - {result_df.index.max().strftime('%Y-%m-%d')}\n")
        
        return result_df
    
    def get_index_history(self, 
                         index_ticker: str,
                         start_date: str, 
                         end_date: str) -> pd.DataFrame:

        url = f"{self.BASE_URL}/history/engines/stock/markets/index/securities/{index_ticker}.json"
        
        all_data = []
        start_index = 0
        
        print(f"   Загрузка индекса {index_ticker}...", end=' ', flush=True)
        
        while True:
            params = {
                'from': start_date,
                'till': end_date,
                'start': start_index,
            }
            
            success = False
            for attempt in range(self.retry_attempts):
                try:
                    response = requests.get(url, params=params, timeout=self.timeout)
                    response.raise_for_status()
                    data = response.json()
                    success = True
                    break
                except Exception as e:
                    if attempt == self.retry_attempts - 1:
                        print(f"✗ Ошибка после {self.retry_attempts} попыток: {e}")
                        return pd.DataFrame()
                    time.sleep(2)
            
            if not success:
                return pd.DataFrame()
            
            if 'history' not in data or not data['history']['data']:
                break
            
            columns = data['history']['columns']
            rows = data['history']['data']
            
            all_data.extend(rows)
            
            if len(rows) < 100:
                break
            
            start_index += 100
            time.sleep(self.delay)
        
        if not all_data:
            print(f"Нет данных")
            return pd.DataFrame()

        df = pd.DataFrame(all_data, columns=columns)
        df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
        df['CLOSE'] = pd.to_numeric(df['CLOSE'], errors='coerce')
        
        df = df[['TRADEDATE', 'CLOSE']].dropna()
        df = df.sort_values('TRADEDATE')
        df = df.drop_duplicates(subset=['TRADEDATE'], keep='last')
        
        df = df.rename(columns={'TRADEDATE': 'Date', 'CLOSE': 'Close'})
        df = df.set_index('Date')
        
        print(f"{len(df)} дней")
        
        return df

# -------------------------------------

def main():
    
    print("\n" + "="*80)
    print("Пасер котировок с МосБиржи".center(80))
    print("MOEX Data Parser v1.0".center(80))
    print("="*80 + "\n")
    
    # Создание парсера с параметрами из панели управления
    parser = MOEXParser(
        timeout=TIMEOUT,
        retry_attempts=RETRY_ATTEMPTS,
        delay=DELAY_BETWEEN_REQUESTS
    )
    
    # Загрузка акций
    stocks_df = parser.get_stocks_history(
        tickers=TICKERS,
        start_date=START_DATE,
        end_date=END_DATE,
        board=BOARD
    )
    
    if stocks_df.empty:
        print("Не удалось загрузить данные по акциям")
        return None, None
    
    # Вывод информации
    print("Загруженные данные:")
    print(f"   Форма: {stocks_df.shape} (дней x акций)")
    print(f"   Колонки: {', '.join(stocks_df.columns)}")
    print(f"\n   Первые 5 дней:")
    print(stocks_df.head())
    print(f"\n   Последние 5 дней:")
    print(stocks_df.tail())
    
    # Простая статистика
    print(f"\nДоходность за период:")
    returns = (stocks_df.iloc[-1] / stocks_df.iloc[0] - 1) * 100
    for ticker in returns.index:
        print(f"   {ticker}: {returns[ticker]:>7.2f}%")
    
    # Загрузка индекса (опционально)
    index_df = None
    if INDEX_TICKER:
        print(f"\nЗагрузка индекса {INDEX_TICKER}:")
        print("-" * 80)
        index_df = parser.get_index_history(
            index_ticker=INDEX_TICKER,
            start_date=START_DATE,
            end_date=END_DATE
        )
        
        if not index_df.empty:
            index_return = (index_df['Close'].iloc[-1] / index_df['Close'].iloc[0] - 1) * 100
            print(f"\nДоходность индекса {INDEX_TICKER}: {index_return:.2f}%")
        else:
            print(f"Не удалось загрузить индекс {INDEX_TICKER}")
    
    print("\n" + "="*80)
    print("Загрузка данных завершена успешно!")
    print("="*80 + "\n")
    
    return stocks_df, index_df


# -------------------------------------

if __name__ == "__main__":
    stocks_data, index_data = main()


                           Пасер котировок с МосБиржи                           
                             MOEX Data Parser v1.0                              


📥 Загрузка данных с Мосбиржи:
   Период: 2024-01-01 - 2024-12-31
   Тикеры: SBER, GAZP, LKOH, YNDX, GMKN, ROSN, NVTK, MTSS
--------------------------------------------------------------------------------
   Загрузка SBER... ✓ 256 дней
   Загрузка GAZP... ✓ 256 дней
   Загрузка LKOH... ✓ 256 дней
   Загрузка YNDX... ✓ 114 дней
   Загрузка GMKN... ✓ 252 дней
   Загрузка ROSN... ✓ 256 дней
   Загрузка NVTK... ✓ 256 дней
   Загрузка MTSS... ✓ 256 дней
--------------------------------------------------------------------------------
Загружено 110 торговых дней для 8 акций
Период фактически: 2024-01-03 - 2024-06-14

Загруженные данные:
   Форма: (110, 8) (дней x акций)
   Колонки: SBER, GAZP, LKOH, YNDX, GMKN, ROSN, NVTK, MTSS

   Первые 5 дней:
              SBER    GAZP    LKOH    YNDX     GMKN    ROSN    NVTK    MTSS
Date      

In [11]:
stocks_data

,SBER,GAZP,LKOH,YNDX,GMKN,ROSN,NVTK,MTSS
Date,,,,,,,,
2024-01-03,274.56,161.44,6803.5,2581.0,16356.00,604.00,1484.8,250.20
2024-01-04,274.12,161.25,6771.0,2568.0,16296.00,601.15,1479.6,250.40
2024-01-05,273.62,161.94,6780.0,2576.8,16280.00,604.05,1480.4,251.60
2024-01-08,276.76,163.05,6766.5,2611.0,16426.00,602.30,1487.2,254.50
2024-01-09,275.28,162.41,6931.0,2607.0,16384.00,600.55,1489.0,261.35
...,...,...,...,...,...,...,...,...
2024-06-07,319.90,122.78,7489.0,4222.0,143.28,571.65,1034.4,295.90
2024-06-10,317.28,119.00,7315.0,4125.6,139.14,563.25,1026.0,289.95
2024-06-11,317.80,117.53,7290.0,4102.0,138.86,562.85,1078.0,288.25


In [13]:
index_data

,Close
Date,
2024-01-03,3130.23
2024-01-04,3136.07
2024-01-05,3136.37
2024-01-08,3158.58
2024-01-09,3155.55
...,...
2024-12-25,2732.83
2024-12-26,2766.57
2024-12-27,2757.45
